In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Mrk-553_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Mrk-553_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     361   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        42   (2094, 964)   float32   


In [3]:
image_cut = image[75:175, 0:1890]
image_error_cut = image_error[75:175, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.01505 
l_HA_z = 6563 * (1+z)
l_HB_z = 4861 * (1+z) 
l_NII_z = 6583 * (1+z)
l_SII_z = 6717 * (1+z)
l_OIII_z = 5007 * (1+z)

l_HA = 6654
l_HB = 4929 
l_NII = 6676
l_SII = 6811
l_OIII = 5077

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = 3.174
sigma = seeing / 2.35 
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/Halpha_fit.csv", index_col = 0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

4.366734152818013
5.4101679858518805
6.506243790291555
6.543789106314056
2.343077231719125
2.3440519836927463
1.3297034604024316
1.352910656696015
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.262850  25.143669  12.472979   5.814737
Green   12.182561  62.101341   7.576148   0.004530
Purple   3.657968  11.193978  71.955314  17.465493
Orange   0.896621   1.561012   7.995559  76.715240


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

18.90547541769704
22.389359343466232
12.822325765070262
12.970945624322903
4.68562801444576
4.687669747663765
3.2036237397644247
3.2445240656386725
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    78.494298  28.572155  12.613113   6.279376
Green   16.115893  58.403255   8.432297   0.008090
Purple   4.350437  11.436473  70.922660  20.865341
Orange   1.039372   1.588117   8.031930  72.847193


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

23.90734410347322
27.532528519783376
21.1045745409105
21.641441519374734
9.466463459588505
9.471207878118246
4.276362135406533
4.318781710255185
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    73.857014  32.373845  13.070778   6.629798
Green   20.021126  52.550906  12.169523   0.011086
Purple   4.966470  13.322123  66.664252  22.659737
Orange   1.155389   1.753126   8.095447  70.699379


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.998530554585597
0.9930898593985995
6.655922699004513
6.452112690799256
1.6406091895784707
1.6422072412400404
3.627412816085022
4.20553071834347
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.930816  87.772721  33.551328   2.033525
Green   14.120763  10.596923  51.975848   1.226558
Purple   3.345645   1.375955  12.343217  92.787971
Orange   0.602775   0.254401   2.129607   3.951946


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.620028645976604
5.580615921862908
12.128106687733794
11.803314816261143
3.3484815561660715
3.3505153282073312
8.110478560345143
9.141997888928874
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.448921  86.298899  33.639463   2.426540
Green   13.023950  11.904745  52.068323   1.554813
Purple   2.988640   1.516201  12.196973  91.779090
Orange   0.538490   0.280155   2.095241   4.239557


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


8.785665104780897
8.697601908055924
18.04400141059794
17.6598927347682
7.155950272434131
7.148691343573825
10.002644458148842
11.073813817358689
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    84.676993  81.412019  34.697433   2.793031
Green   12.145087  16.369285  51.496229   1.828916
Purple   2.692727   1.873010  11.813658  90.496413
Orange   0.485194   0.345686   1.992680   4.881640


## NII

In [16]:
mask_NII = (wv > l_NII-8)*(wv < l_NII+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.874486669590774
0.8706262589278506
5.130070519636471
5.30259398702816
2.104488913253846
2.1192078493973505
5.314435651300981
6.258084041379977
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    82.145722  88.596020  30.095585   1.586679
Green    9.755475   7.772401  52.442262   1.187788
Purple   6.293770   2.847600  14.236037  93.505240
Orange   1.805033   0.783980   3.226115   3.720293


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

4.749964472612269
4.733698015743036
9.54262242924326
9.81864463292017
4.270734640248334
4.311557954031266
11.384742469282983
13.057129926730534
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.237373  87.472712  30.319144   1.979006
Green    9.275619   8.610914  52.361877   1.520823
Purple   5.820438   3.071911  14.132009  92.302505
Orange   1.666570   0.844463   3.186970   4.197666


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.221517690104865
7.221539658304418
14.77149543124625
15.09349794850898
8.690939487477957
8.787342917846646
13.998824697046874
15.739591923062992
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    84.222025  83.435557  32.705727   2.285481
Green    8.851264  11.950692  49.824967   1.773777
Purple   5.387007   3.627320  14.307385  91.114188
Orange   1.539704   0.986432   3.161921   4.826554


## SII

In [20]:
mask_SII = (wv > l_SII-8)*(wv < l_SII+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5532038865464582
0.5514254315443956
2.5746410974896783
2.6362378985982797
1.3378007253006061
1.342264921800749
3.714094292495415
4.060546538822447
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    73.539034  81.848233  30.226419   1.502612
Green   18.092782  13.413020  51.655942   2.896615
Purple   5.349242   3.155711  13.404311  91.320392
Orange   3.018941   1.583036   4.713328   4.280382


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.911538408595828
2.90454448747635
4.8871950490291285
4.98544304494113
2.6973443174592937
2.7095282741104807
7.918484110083157
8.531944040722898
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    74.571901  80.660192  30.540118   1.885818
Green   17.443698  14.331663  51.371140   3.590387
Purple   5.110167   3.338000  13.404983  89.654334
Orange   2.874234   1.670145   4.683758   4.869461


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.291496446803145
4.2908712331353325
7.848137659538294
7.962251543441321
5.4503473904499815
5.479056452437378
9.765878641210927
10.40019713584062
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    75.539945  76.783666  32.646055   2.171635
Green   16.840327  17.526785  48.966275   4.120060
Purple   4.883301   3.820557  13.718118  88.132972
Orange   2.736427   1.868992   4.669551   5.575333


## OIII

In [24]:
mask_OIII = (wv > l_OIII-8)*(wv < l_OIII+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.2515532179364304
1.2398667961742404
9.99846522549739
8.480604172623949
1.6760520718442098
1.6778683453096834
4.011037770399565
4.73081802143012
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    86.892280  92.717031  51.100073   4.224125
Green    8.553603   5.759648  30.531888   0.036997
Purple   3.207361   1.084081  13.787020  90.144885
Orange   1.346757   0.439240   4.581019   5.593993


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

7.22173347682586
6.996977427979974
17.72751000925801
15.320525643754399
3.387031731547341
3.390652391490042
8.877749274380776
10.163782289567159
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    88.270195  91.811974  51.513961   5.009195
Green    7.765384   6.464248  30.189957   0.071855
Purple   2.793423   1.227309  13.751289  88.806119
Orange   1.170998   0.496469   4.544793   6.112831


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

11.657579564505124
10.996119997881111
24.90579238344399
22.15487118821416
7.080909942701042
7.087738510581152
11.024598850632335
12.36298268934117
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    89.494767  89.329855  53.364616   5.698693
Green    7.049619   8.428058  28.663519   0.103692
Purple   2.436224   1.601305  13.581566  87.248961
Orange   1.019390   0.640782   4.390299   6.948654
